# Mushroom Dataset - Exploration

Exploratory analysis of the Kaggle Mushroom Classification dataset (UCI-style):
8,124 instances, 22 categorical physical features, binary target `class`
(`e` = edible, `p` = poisonous).

This notebook is for understanding the data only. The production training
workflow lives in `train_mushroom.py` at the project root.

**Safety note:** this is an educational prototype, not a certified mushroom
identification system. Nothing here establishes that a mushroom is safe to eat.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Resolve the CSV relative to this notebook, with a fallback for the
# repository's current layout (data/mushroom/mushrooms.csv).
notebook_dir = Path.cwd()
candidates = [
    notebook_dir.parent / "data" / "mushrooms.csv",
    notebook_dir.parent / "data" / "mushroom" / "mushrooms.csv",
]
csv_path = next((p for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(f"Dataset not found. Searched: {candidates}")

df = pd.read_csv(csv_path)
print(f"Loaded {csv_path}")
print("Shape:", df.shape)
df.head()

: 

## Columns and dtypes

Every column is a short categorical code string. The target is `class`.

In [ ]:
print("Columns:", list(df.columns))
print("\nTarget column: class")
df.dtypes.value_counts()

## Missing-value analysis

pandas does not treat `?` as missing. In this dataset the marker appears in
`stalk-root` and must be handled explicitly.

In [ ]:
audit = pd.DataFrame(
    {
        "n_categories": df.nunique(dropna=False),
        "pandas_nulls": df.isna().sum(),
        "question_mark_markers": df.eq("?").sum(),
    }
)
audit

## Class distribution

The two classes are almost balanced (roughly 52% edible / 48% poisonous).

In [ ]:
label_names = {"e": "edible", "p": "poisonous"}
class_counts = df["class"].value_counts().rename(index=label_names)
print(class_counts)

ax = class_counts.plot.bar(rot=0, color=["tab:green", "tab:red"], title="Class distribution")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## Categorical feature distributions

Selected features that matter most in practice. `veil-type` is constant
across all rows and carries no information.

In [ ]:
plot_features = ["odor", "cap-color", "gill-color", "stalk-root"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.ravel(), plot_features):
    sns.countplot(data=df, x=feature, order=sorted(df[feature].unique()), ax=ax, color="tab:blue")
    ax.set_title(f"Distribution of {feature}")
plt.tight_layout()
plt.show()

## poisonous rate by feature (target-aware view)

How the poisonous fraction varies with a feature. This is exploratory only;
the training script keeps a stratified held-out test set that is never used
for decisions made here.

In [ ]:
df["is_poisonous"] = df["class"].eq("p").astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, feature in zip(axes, ["odor", "stalk-root"]):
    rates = df.groupby(feature)["is_poisonous"].mean().sort_values()
    rates.plot.barh(ax=ax, color="tab:red", title=f"Poisonous fraction by {feature}")
    ax.set_xlim(0, 1)
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

## Summary of findings

- 8,124 rows x 23 columns (22 features + target `class`); all features are
  categorical code strings.
- Class balance is close to even: 4,208 edible vs 3,916 poisonous.
- `stalk-root` contains 2,480 `?` markers that pandas does not count as null;
  no other column has missing values.
- `veil-type` has a single category (`p`) and provides no signal.
- `odor` is the strongest single separator: several odor codes (e.g. `f`,
  `c`, `p`) map to 100% poisonous, while others (e.g. `n`, `a`) are
  essentially edible-only. `stalk-root` and `gill-color` also separate
  the classes well.
- Because features like `odor` are near-deterministic in this curated
  dataset, models reach near-perfect held-out scores here. That does **not**
  imply reliability on real wild samples, which vary by region, season, and
  observer.

Next step: run `python train_mushroom.py` at the project root to train and
evaluate the production Random Forest pipeline.